# 08 · Academic genealogy

**Method** (`chemkg.genealogy`)
- Edges: P184 (doctoral advisor, on the student) ∪ P185 (doctoral student, on the advisor). The lineage pass in `enrich.py` walks advisor chains *upward* beyond the chemist set (up to 8 rounds), so families aren't cut off at a physicist or physician.
- Cycles (data errors) are broken deterministically and reported.
- Metrics: descendants, chemist-descendants, generations, PageRank on the reversed graph (credit flows to mentors), Nobel laureates among descendants, Louvain communities (seed 42).
- **Family trees** are roots (no recorded advisor), ranked by the number of descendants who are chemists in this dataset.

**Caveat:** advisor links are sparse and skew toward famous labs and the German/American research-university tradition.

In [1]:
import json, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from chemkg.data import load, analysable
from chemkg import countries
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 70)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
ds = load()
P, A = ds.people, analysable(ds.people)
NAMES = countries.load_iso3().set_index("iso3").name.to_dict()
print(f"{len(P):,} people, {len(A):,} analysable; enrichment loaded: {ds.enriched}")

11,903 people, 11,860 analysable; enrichment loaded: True


In [2]:
from chemkg import genealogy
g, adv, removed = genealogy.build_graph(ds)
m = genealogy.metrics(g)
print(f"{g.number_of_nodes():,} people, {g.number_of_edges():,} advisor links, {len(removed)} cycle edges removed")
print(f"in dataset: {m.in_dataset.sum():,}; with ≥1 recorded student: {(m.students > 0).sum():,}")

6,322 people, 5,559 advisor links, 3 cycle edges removed
in dataset: 2,372; with ≥1 recorded student: 2,540


In [3]:
trees = genealogy.family_trees(g, m)
pd.DataFrame([{k: t[k] for k in ("name", "birth_year", "chem_descendants", "descendants", "generations", "nobel_descendants")} for t in trees])

,name,birth_year,chem_descendants,descendants,generations,nobel_descendants
0,Vittorino da Feltre,1378.0,495,1254,26,97
1,Georg Balthasar Metzger,1623.0,436,1268,19,92
2,Stephen Whisson,1710.0,79,243,16,28
3,Ludolph van Ceulen,1540.0,58,473,22,12
4,Josiah P. Cooke,1827.0,56,86,7,4
5,Christian Gottlieb Ludwig,1709.0,48,134,13,14
6,Nikolaus Poda von Neuhaus,1723.0,26,99,16,4
7,Hubert Anson Newton,1830.0,17,87,9,1
8,Geoffrey Wilkinson,1921.0,16,33,5,1
9,Eduard Mulder,1832.0,13,44,7,1


In [4]:
m[m.in_dataset].nlargest(15, "pagerank")[["name", "birth_year", "students", "descendants", "chem_descendants", "nobel_descendants", "pagerank"]]

,name,birth_year,students,descendants,chem_descendants,nobel_descendants,pagerank
69,Justus von Liebig,1803.0,18,477,262,54,0.005238
60,Friedrich Stromeyer,1776.0,3,587,297,50,0.005220
68,Karl Wilhelm Gottlob Kastner,1783.0,1,478,263,54,0.004514
214,Linus Pauling,1901.0,26,122,43,8,0.004372
125,Robert Bunsen,1811.0,30,246,147,29,0.004017
67,Johann Friedrich August Göttling,1755.0,1,479,264,54,0.003900
228,Peter Debye,1884.0,37,98,6,1,0.003476
207,Arthur Amos Noyes,1866.0,4,129,48,9,0.003466
66,Johann Christian Wiegleb,1732.0,1,480,265,54,0.003409
57,Herman Boerhaave,1668.0,16,625,254,43,0.003304


In [5]:
m[m.in_dataset].nlargest(15, "nobel_descendants")[["name", "birth_year", "students", "chem_descendants", "nobel_descendants", "generations"]]

,name,birth_year,students,chem_descendants,nobel_descendants,generations
31,Daniel Sennert,1572.0,1,308,65,20
32,Werner Rolfinck,1599.0,2,307,65,19
24,Franciscus Sylvius,1614.0,3,293,54,19
41,Philipp Friedrich Gmelin,1721.0,1,320,54,15
42,Johann Friedrich Gmelin,1748.0,3,319,54,14
62,Friedrich Hoffmann,1660.0,1,268,54,17
64,Ernst Anton Nicolai,1722.0,1,267,54,15
66,Johann Christian Wiegleb,1732.0,1,265,54,13
67,Johann Friedrich August Göttling,1755.0,1,264,54,12
68,Karl Wilhelm Gottlob Kastner,1783.0,1,263,54,11


In [6]:
comm = m.groupby("community").agg(size=("qid", "size"), nobel=("nobel", "sum"),
     exemplar=("name", lambda s: ", ".join(s.iloc[:3])), median_birth=("birth_year", "median")).sort_values("size", ascending=False)
comm.head(10)

,size,nobel,exemplar,median_birth
community,,,,
0,147,6,"Abraham Gotthelf Kästner, Georg Christoph Lichtenberg, Karl Christ...",1842.5
1,127,7,"Rudolf Wagner, Julius Vogel, Carl Schmidt",1877.0
2,123,6,"Georg Balthasar Metzger, Elias Rudolph Camerarius Sr., Friedrich L...",1763.0
3,111,8,"Johan Afzelius, Jöns Jacob Berzelius, Heinrich Rose",1906.0
4,106,8,"Friedrich Wöhler, Robert Bunsen, Georg Städeler",1860.0
5,98,11,"Stephen Whisson, Thomas Postlethwaite, Edward Waring",1901.0
6,95,0,"Margaret Brimble, John Llewellyn, David Hall",1931.0
7,94,3,"Peter Debye, Paul Scherrer, Max Le Blanc",1890.0
8,94,2,"Hubert Anson Newton, Eliakim Hastings Moore, George David Birkhoff",1913.0


In [7]:
print(genealogy.lineage_path(g, *[m.loc[m.name == n, "qid"].iloc[0] for n in ("Linus Pauling", "Arnold Sommerfeld")]) if {"Linus Pauling", "Arnold Sommerfeld"} <= set(m.name) else "names not both present")

['Q48983', 'Q28935564', 'Q106733', 'Q107422', 'Q102077191', 'Q2149135', 'Q65201', 'Q61766', 'Q41585', 'Q77203', 'Q77078']


### Reading the result
- One giant connected family contains most recorded links. It runs through 19th-century German chemistry (Liebig: 262 chemist descendants and 54 Nobel laureates; Bunsen: 147; Wöhler: 141) and, via the lineage pass, back to Renaissance and early-modern physicians and scholars.
- Nobel laureates cluster in these lineages: being in a large, well-documented family and winning a Nobel are correlated. Part of that is documentation bias, since famous people get their advisors recorded.